# 🩺 Diabetes Clinical RAG Chatbot — Deploy on Colab

This notebook deploys your Diabetes Clinical Assistant with a **public URL** using ngrok.

**Run each cell in order (1 → 5).**

---

## Cell 1: Mount Google Drive
This connects your Google Drive so Colab can access your project files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === CONFIGURE YOUR PATH HERE ===
# Change this if your folder name or location is different
PROJECT_PATH = '/content/drive/MyDrive/RAG_Chatbot/chatbot--Diabetes-Mellitus-main'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f'✅ Project found at: {PROJECT_PATH}')
    print(f'📂 Files: {os.listdir(".")}')
else:
    print(f'❌ Project NOT found at: {PROJECT_PATH}')
    print(f'   Please upload your project to Google Drive first.')
    print(f'   Expected structure: My Drive/RAG_Chatbot/chatbot--Diabetes-Mellitus-main/')

## Cell 2: Install Dependencies
Installs all required Python packages. Takes ~2-3 minutes.

In [ ]:
%%capture install_output
!pip install -q langchain>=0.3.0 langchain-community>=0.3.0 langchain-core>=0.3.0
!pip install -q langchain-chroma>=0.1.0 langchain-text-splitters>=0.3.0
!pip install -q pypdf>=3.17.0 pymupdf>=1.23.0
!pip install -q sentence-transformers>=2.2.0 transformers>=4.36.0
!pip install -q fastapi>=0.115.0 uvicorn>=0.30.0 pydantic>=2.8.0
!pip install -q streamlit>=1.35.0 requests>=2.32.0
!pip install -q python-dotenv>=1.0.0 pandas>=2.2.0 numpy>=1.26.0
!pip install -q pyngrok

print('✅ All dependencies installed!')

## Cell 3: Set Your ngrok Auth Token

Get your **free** token from: https://dashboard.ngrok.com/get-started/your-authtoken

Paste it below when prompted.

In [ ]:
from getpass import getpass

NGROK_TOKEN = getpass('🔑 Paste your ngrok auth token: ')

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok authenticated!')
else:
    print('❌ No token provided. Get one at https://dashboard.ngrok.com/signup')

## Cell 4: Run Data Ingestion
Processes your PDF files and builds the ChromaDB vector index. Takes ~1-2 minutes.

In [ ]:
import sys
import os
from pathlib import Path

# Make sure we're in the project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

# Check for PDFs
pdf_dir = Path('data/raw_pdfs')
pdfs = list(pdf_dir.glob('*.pdf'))
print(f'📄 Found {len(pdfs)} PDF(s): {[p.name for p in pdfs]}')

if not pdfs:
    print('❌ No PDFs found! Make sure your PDFs are in data/raw_pdfs/')
else:
    # Check if index already exists
    chroma_dir = Path('data/chroma_db')
    if chroma_dir.exists() and any(chroma_dir.iterdir()):
        print('📦 ChromaDB index already exists. Skipping ingestion.')
        print('   (Delete data/chroma_db/ to force re-ingestion)')
    else:
        print('🔄 Running ingestion pipeline...')
        from core.ingest import main as ingest_main
        ingest_main()
        print('\n✅ Ingestion complete!')

## Cell 5: Start the App & Get Your Public URL 🎉

This starts both FastAPI and Streamlit, then creates a public ngrok tunnel.

**Your public URL will be printed below — share it with anyone!**

In [ ]:
import subprocess
import time
import sys
import os
import requests

os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

# Kill any existing processes
!pkill -f uvicorn 2>/dev/null || true
!pkill -f streamlit 2>/dev/null || true

from pyngrok import ngrok

# Close any existing tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

print('🚀 Starting FastAPI backend on port 8000...')
api_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for API to be ready
print('⏳ Waiting for API to load models (this may take 1-2 minutes)...')
for i in range(120):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ API is ready! ({i+1}s)')
            break
    except:
        pass
    time.sleep(1)
else:
    print('⚠️ API may still be loading. Continuing anyway...')

print('\n🎨 Starting Streamlit frontend on port 8501...')
ui_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app/ui.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.address', '0.0.0.0',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

# Create ngrok tunnel to Streamlit
print('🌐 Creating public tunnel...')
public_url = ngrok.connect(8501, 'http')

print()
print('═' * 56)
print('🎉 YOUR CHATBOT IS LIVE!')
print('═' * 56)
print()
print(f'🌐 Public URL: {public_url.public_url}')
print()
print('Share this link with anyone!')
print('═' * 56)
print()
print('ℹ️  Keep this notebook running to keep the chatbot alive.')
print('    The URL will stop working when you close this notebook.')
print()
print('📡 Local endpoints (for debugging):')
print('    API:  http://localhost:8000')
print('    UI:   http://localhost:8501')
print('    Docs: http://localhost:8000/docs')

---

## 🛑 Stop the App

Run this cell to shut everything down:

In [ ]:
from pyngrok import ngrok

# Close tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Kill processes
!pkill -f uvicorn 2>/dev/null || true
!pkill -f streamlit 2>/dev/null || true

print('✅ All services stopped.')